# Handle Missing Values and Normalize Numerical Features

This notebook demonstrates how to handle missing values and normalize numerical features in a dataset. We show the code and results before and after each step.

## 1. Import Required Libraries
We will use pandas for data manipulation and scikit-learn for preprocessing.

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler


## 2. Load Sample Dataset with Missing Values
We will load the materials CSV file and display the first few rows to observe missing values.

In [2]:
# Load materials CSV
materials_df = pd.read_csv('ecopack_materials.csv')

# Show first 10 rows
materials_df.head(10)

,Material_Type,Strength_PSI,Weight_Capacity_KG,Biodegradability_%,CO2_Emission_Score_%,Recyclability_%,Cost_per_KG_USD
0,Molded Pulp,1746.19,5.18,89.54,31.97,87.23,0.79
1,Aluminum Packaging,11878.75,22.12,0.19,66.19,98.92,2.69
2,Kraft Paper,5699.76,6.78,91.07,30.28,87.56,0.67
3,Kraft Paper,4649.23,9.66,NaN,47.86,87.75,0.76
4,Starch-Based Packaging,2103.02,2.07,NaN,25.62,69.94,1.77
5,PLA Bioplastic,8053.78,1.19,68.96,53.47,58.57,3.41
6,Plywood (Lightweight),14392.26,78.08,54.43,54.87,50.18,2.03
7,Bamboo Fiber,6288.65,28.32,85.29,22.44,85.77,1.63
8,Plywood (Lightweight),10909.85,77.26,60.88,41.17,56.53,3.22
9,PHA Bioplastic,7177.07,2.14,81.00,43.87,48.98,2.50


## 3. Handle Missing Values (Before and After)
We will display missing values, apply imputation, and show the results.

In [3]:
# Show missing values before
materials_df.isnull().sum()

# Numerical columns
num_cols = [
    'Strength_PSI',
    'Weight_Capacity_KG',
    'Biodegradability_%',
    'CO2_Emission_Score_%',
    'Recyclability_%',
    'Cost_per_KG_USD'
]

# Group-wise mean imputation based on Material_Type
for col in num_cols:
    materials_df[col] = materials_df.groupby('Material_Type')[col] \
                                     .transform(lambda x: x.fillna(x.mean()))

# Fallback: if any value still missing (e.g., whole group was NaN)
materials_df[num_cols] = materials_df[num_cols].fillna(materials_df[num_cols].mean())

# Show missing values after
materials_df.isnull().sum()


Material_Type           0
Strength_PSI            0
Weight_Capacity_KG      0
Biodegradability_%      0
CO2_Emission_Score_%    0
Recyclability_%         0
Cost_per_KG_USD         0
dtype: int64

In [4]:
# Show first 10 rows after imputation
materials_df.head(10)

,Material_Type,Strength_PSI,Weight_Capacity_KG,Biodegradability_%,CO2_Emission_Score_%,Recyclability_%,Cost_per_KG_USD
0,Molded Pulp,1746.19,5.18,89.540000,31.97,87.23,0.79
1,Aluminum Packaging,11878.75,22.12,0.190000,66.19,98.92,2.69
2,Kraft Paper,5699.76,6.78,91.070000,30.28,87.56,0.67
3,Kraft Paper,4649.23,9.66,85.970508,47.86,87.75,0.76
4,Starch-Based Packaging,2103.02,2.07,94.806545,25.62,69.94,1.77
5,PLA Bioplastic,8053.78,1.19,68.960000,53.47,58.57,3.41
6,Plywood (Lightweight),14392.26,78.08,54.430000,54.87,50.18,2.03
7,Bamboo Fiber,6288.65,28.32,85.290000,22.44,85.77,1.63
8,Plywood (Lightweight),10909.85,77.26,60.880000,41.17,56.53,3.22
9,PHA Bioplastic,7177.07,2.14,81.000000,43.87,48.98,2.50


In [5]:
print(num_cols)

['Strength_PSI', 'Weight_Capacity_KG', 'Biodegradability_%', 'CO2_Emission_Score_%', 'Recyclability_%', 'Cost_per_KG_USD']


In [6]:
# Assuming your DataFrame is named 'df' and you've already handled missing values
materials_df.to_csv('cleaned_dataset.csv', index=False)

In [7]:
# Load the cleaned dataset after missing value handling
materials_df = pd.read_csv('cleaned_dataset.csv')

# Feature Engineering
# 1. CO2 Impact Index (lower CO2 emission and higher recyclability is better)
materials_df['CO2_Impact_Index'] = (
    (1 - (materials_df['CO2_Emission_Score_%'] - materials_df['CO2_Emission_Score_%'].min()) /
     (materials_df['CO2_Emission_Score_%'].max() - materials_df['CO2_Emission_Score_%'].min())) * 0.7 +
    (materials_df['Recyclability_%'] / 100) * 0.3
)

# 2. Cost Efficiency Index (higher weight capacity and strength, lower CO2 emission)
materials_df['Cost_Efficiency_Index'] = (
    (materials_df['Weight_Capacity_KG'] / materials_df['Weight_Capacity_KG'].max()) * 0.4 +
    (materials_df['Strength_PSI'] / materials_df['Strength_PSI'].max()) * 0.4 +
    (1 - (materials_df['CO2_Emission_Score_%'] - materials_df['CO2_Emission_Score_%'].min()) /
     (materials_df['CO2_Emission_Score_%'].max() - materials_df['CO2_Emission_Score_%'].min())) * 0.2
)

# 3. Material Suitability Score (composite: strength, biodegradability, recyclability, low CO2)
materials_df['Material_Suitability_Score'] = (
    (materials_df['Strength_PSI'] / materials_df['Strength_PSI'].max()) * 0.3 +
    (materials_df['Biodegradability_%'] / materials_df['Biodegradability_%'].max()) * 0.2 +
    (materials_df['Recyclability_%'] / 100) * 0.2 +
    (1 - (materials_df['CO2_Emission_Score_%'] - materials_df['CO2_Emission_Score_%'].min()) /
     (materials_df['CO2_Emission_Score_%'].max() - materials_df['CO2_Emission_Score_%'].min())) * 0.3
)

# 4. Data Quality Validation: Summary statistics
summary_stats = materials_df.describe(include='all')
missing_values = materials_df.isnull().sum()

print('Summary Statistics:')
print(summary_stats)
print('\nMissing Values:')
print(missing_values)

# Show head of engineered features
display(materials_df[['Material_Type','CO2_Impact_Index','Cost_Efficiency_Index','Material_Suitability_Score']].head())

Summary Statistics:
       Material_Type  Strength_PSI  Weight_Capacity_KG  Biodegradability_%  \
count            900    900.000000          900.000000          900.000000   
unique            15           NaN                 NaN                 NaN   
top      Kraft Paper           NaN                 NaN                 NaN   
freq              65           NaN                 NaN                 NaN   
mean             NaN   6193.771514           16.020745           76.648074   
std              NaN   3456.709969           12.642904           27.602530   
min              NaN   1027.400000            1.020000            0.120000   
25%              NaN   3692.002500            7.455000           73.252500   
50%              NaN   5493.055000           13.055000           86.755000   
75%              NaN   7640.442500           20.765000           94.027500   
max              NaN  19441.720000           78.080000           99.890000   

        CO2_Emission_Score_%  Recyclability

,Material_Type,CO2_Impact_Index,Cost_Efficiency_Index,Material_Suitability_Score
0,Molded Pulp,0.768532,0.207275,0.597900
1,Aluminum Packaging,0.501229,0.416136,0.469148
2,Kraft Paper,0.784455,0.301081,0.669030
3,Kraft Paper,0.629685,0.249838,0.576415
4,Starch-Based Packaging,0.772771,0.214716,0.603418


In [8]:
# Save the engineered dataset to a new CSV file
materials_df.to_csv('engineered_dataset.csv', index=False)

In [9]:
# Data Quality Validation: Summary statistics for engineered dataset
import pandas as pd
engineered_df = pd.read_csv('engineered_dataset.csv')

summary_stats = engineered_df.describe(include='all')
missing_values = engineered_df.isnull().sum()

print('Summary Statistics:')
print(summary_stats)
print('\nMissing Values:')
print(missing_values)

Summary Statistics:
       Material_Type  Strength_PSI  Weight_Capacity_KG  Biodegradability_%  \
count            900    900.000000          900.000000          900.000000   
unique            15           NaN                 NaN                 NaN   
top      Kraft Paper           NaN                 NaN                 NaN   
freq              65           NaN                 NaN                 NaN   
mean             NaN   6193.771514           16.020745           76.648074   
std              NaN   3456.709969           12.642904           27.602530   
min              NaN   1027.400000            1.020000            0.120000   
25%              NaN   3692.002500            7.455000           73.252500   
50%              NaN   5493.055000           13.055000           86.755000   
75%              NaN   7640.442500           20.765000           94.027500   
max              NaN  19441.720000           78.080000           99.890000   

        CO2_Emission_Score_%  Recyclability